In [ ]:
import os

# Option 1: Set directly (for learning only — never commit real keys!)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# Option 2: Use .env file (recommended)
# uv add python-dotenv
# from dotenv import load_dotenv
# load_dotenv()


In [ ]:
from agents import Agent, Runner

# Step 1: Create an Agent
agent = Agent(
    name="Greeter",  # Give it a name
    instructions="You are a friendly assistant. Be concise.",  # Its personality/role
)

# # Step 2: Run it (sync version — good for scripts)
# result = Runner.run_sync(agent, "Hello! What can you do?")

# # Step 3: Get the output
# print(result.final_output)


In [ ]:
import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from agents.tracing import set_tracing_disabled

# Disable tracing (optional)
set_tracing_disabled(True)

model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",  # required but ignored by Ollama
        base_url="http://localhost:11434/v1",
    ),
)

agent = Agent(
    name="Greeter",
    instructions="You are a friendly assistant. Be concise.",
    # model="gpt-5.4-mini"
    model=model,
)

# In Jupyter, use await directly (no asyncio.run needed)
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)


In [ ]:
from agents import Agent, Runner

agent = Agent(model=model, name="Analyst", instructions="You analyze topics briefly.")

result = await Runner.run(
    agent,
    "Why is Python popular for AI?",
)

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")  # The text response
print(f"Last Agent:    {result.last_agent.name}")  # Which agent finished
print(f"New Items:     {len(result.new_items)}")  # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")  # Raw LLM responses
print("=" * 50)


In [ ]:
from agents import Agent, Runner

agent = Agent(
    model=model,
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [
    {"role": "user", "content": "What about reverse sort?"}
]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")


In [ ]:
from agents import Agent, Runner

agent = Agent(model=model, name="Streamer", instructions="Explain briefly.")

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, "delta"):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")


In [ ]:
import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from agents.tracing import set_tracing_disabled

# Disable tracing (no external telemetry)
set_tracing_disabled(True)

# ── Model setup ───────────────────────────────────────────────────────────────
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",  # make sure this exists in ollama list
    openai_client=ollama_client,
)

# ── Create Agent ──────────────────────────────────────────────────────────────
agent = Agent(
    name="Lahore Guide",
    instructions=(
        "You are an expert local guide for Lahore, Pakistan. "
        "Provide rich, vivid, and practical recommendations about history, culture, food, and places. "
        "Mention specific locations, timings, and tips. "
        "Keep responses engaging but informative."
    ),
    model=model,
)


# ── Main conversation flow ─────────────────────────────────────────────────────
async def main():
    history = None

    # ── Turn 1 ────────────────────────────────────────────────────────────────
    print("=== Turn 1 ===")
    result = await Runner.run(
        agent, "What are the must-visit historical sites in Lahore?"
    )
    print(result.final_output)

    # Save history
    history = result.to_input_list()

    # ── Turn 2 ────────────────────────────────────────────────────────────────
    print("\n=== Turn 2 ===")
    history.append(
        {"role": "user", "content": "Which of those is best to visit at sunset?"}
    )

    result = await Runner.run(agent, history)
    print(result.final_output)

    history = result.to_input_list()

    # ── Turn 3 ────────────────────────────────────────────────────────────────
    print("\n=== Turn 3 ===")
    history.append(
        {"role": "user", "content": "Any nearby street food spots I shouldn't miss?"}
    )

    result = await Runner.run(agent, history)
    print(result.final_output)


# ── Safe runner (works in Jupyter + script) ────────────────────────────────────
if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError:
        # Fix for Jupyter / already running loop
        import nest_asyncio

        nest_asyncio.apply()
        asyncio.get_event_loop().run_until_complete(main())
